# EASE + GF-CF (baseline bổ sung) trên Kaggle CPU

Hai baseline nghiệm đóng, không train bằng gradient: **EASE** chính xác (Cholesky tại chỗ) và **GF-CF**
(bộ lọc đồ thị, SVD k=256 bằng đúng thư viện `sparsesvd` của tác giả, được cài và kiểm tra ở cell đầu). Đủ catalog 44.8k item, 4 mật độ × seed 42, một cấu hình cố định mỗi
model (`configs/ease.yaml`, `configs/gfcf.yaml`). EASE cần khoảng **9 GB RAM trống**; GF-CF nhẹ. Không dùng GPU.

**Cách chạy**
1. Settings: Accelerator **None (CPU)**, Internet **On**.
2. **Save Version → Save & Run All (Commit)**.
3. Tải **`ket_qua_ease_gfcf.zip`** ở cell cuối (hoặc tab Output), giải nén vào thư mục repo trên máy:
   nó chỉ chứa `results/ease/...` và `results/gfcf/...`, không ghi đè kết quả của 5 mô hình kia.
   Sau đó chạy `python scripts\summarize_results.py` trên máy để cập nhật `results/summary.csv`.

Run đã xong (có `metrics.json`) được bỏ qua. Thiếu RAM thì EASE dừng với `MemoryError` (không thu nhỏ catalog).


In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from IPython.display import FileLink, display
import psutil

REPO_URL = "https://github.com/tuyenda2011/SGKR-SPHR-RecSys.git"   # repo có run_ease.py, run_gfcf.py
CODE = Path("/tmp/SGKR-SPHR-RecSys")          # code + data (not kept in the output)
WORK = Path("/kaggle/working")
RESULTS = WORK / "results"                    # kept as the notebook output
MODELS = ["gfcf", "ease"]                     # GF-CF first: fast, so its results exist even if EASE runs out of RAM
DENSITIES = ["0.25", "0.5", "0.75", "1.0"]
TOTAL_RUNS = len(MODELS) * len(DENSITIES)
print(f"RAM trống {psutil.virtual_memory().available / 2**30:.1f} GiB (EASE cần ~8.5 GiB), CPU {os.cpu_count()}")

# GF-CF uses the authors' SVD library (sparsesvd, SVDLIBC; source-only, built here). Check it on a small matrix;
# if it cannot be built, GF-CF is skipped with a message instead of silently switching SVD method.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sparsesvd"])
check = subprocess.run([sys.executable, "-c",
    "import numpy as np, scipy.sparse as sp; from sparsesvd import sparsesvd; "
    "a = sp.random(300, 200, density=0.05, format='csc', random_state=0); "
    "u, s, vt = sparsesvd(a, 10); ref = np.linalg.svd(a.toarray(), compute_uv=False)[:10]; "
    "assert vt.shape == (10, 200) and np.allclose(np.sort(s)[::-1], ref, rtol=1e-5), (s, ref); print('sparsesvd ok')"],
    capture_output=True, text=True)
print(check.stdout.strip() or check.stderr.strip()[-800:])
if check.returncode and "gfcf" in MODELS:
    MODELS.remove("gfcf")
    print("sparsesvd không dùng được: BỎ QUA GF-CF ở phiên này (EASE vẫn chạy).")

In [ ]:
if not CODE.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE)], check=True)
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True, check=True).stdout.strip()
(WORK / "code_commit.txt").write_text(commit + "\n")
print("code commit:", commit)
RESULTS.mkdir(parents=True, exist_ok=True)
# Finished runs of an earlier session (Add Input -> this notebook's output) are reused, not repeated.
for run in sorted(glob.glob("/kaggle/input/**/results/*/d*/seed*", recursive=True)):
    run = Path(run)
    if run.parents[1].name not in MODELS:
        continue
    target = RESULTS / run.relative_to(run.parents[2])
    if (run / "metrics.json").exists() and not (target / "metrics.json").exists():
        shutil.rmtree(target, ignore_errors=True)
        shutil.copytree(run, target)
print(f"{len(list(RESULTS.glob('*/d*/seed*/metrics.json')))}/{TOTAL_RUNS} runs đã xong")

In [ ]:
start = time.time()
with open(WORK / "closed_form.log", "a") as log:
    for model in MODELS:
        for density in DENSITIES:
            t0 = time.time()
            code = subprocess.run([sys.executable, "-u", "scripts/train_model.py", "--model", model, "--density", density,
                                   "--results", str(RESULTS)], cwd=CODE, stdout=log, stderr=subprocess.STDOUT).returncode
            print(f"{model} d{round(float(density) * 100)}: exit {code}, {(time.time() - t0) / 60:.1f} phút", flush=True)
print(f"tổng {(time.time() - start) / 60:.1f} phút")
print("".join(open(WORK / "closed_form.log", encoding="utf-8", errors="replace").readlines()[-8:]))

In [ ]:
import json

finished = sorted(RESULTS.glob("*/d*/seed*/metrics.json"))
for path in finished:
    run = json.loads(path.read_text(encoding="utf-8"))
    print(f"{run['model']:5s} d{round(run['density'] * 100):3d}: NDCG@20={run['val_metrics']['NDCG@20']:.5f} "
          f"Recall@20={run['val_metrics']['Recall@20']:.5f} fit={run['train_seconds'] / 60:.1f} phút")

# Only results/{ease,gfcf}/**: extracting it locally cannot overwrite the other models.
# logs/<model>/model.npz is a hard link of checkpoint.npz, so it is left out of the archive.
archive = WORK / "ket_qua_ease_gfcf.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for model in MODELS:
        for item in RESULTS.glob(f"{model}/**/*"):
            if item.is_file() and item.name != "model.npz":
                zf.write(item, Path("results") / item.relative_to(RESULTS))
    zf.write(WORK / "closed_form.log", "logs/closed_form.log")
print(f"ket_qua_ease_gfcf.zip: {archive.stat().st_size / 2**20:.1f} MiB")
display(FileLink(archive.name))
print("Xong." if len(finished) == TOTAL_RUNS else f"Chưa đủ {TOTAL_RUNS} run: xem closed_form.log (EASE thiếu RAM -> MemoryError).")